$\newcommand{\ket}[1]{\left\lvert #1\right\rangle}
\newcommand{\bra}[1]{\left\langle #1\right\rvert}
\newcommand{\braket}[1]{\left\langle #1\right\rangle}
\newcommand{\proj}[1]{\lvert #1\rangle\!\langle #1\rvert}
\newcommand{\tr}{\operatorname{tr}}\newcommand{\Tr}{\operatorname{Tr}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\rank}{\operatorname{rank}}\newcommand{\diag}{\operatorname{diag}}
\newcommand{\poly}{\operatorname{poly}}\newcommand{\polylog}{\operatorname{polylog}}
\newcommand{\norm}[1]{\lVert #1\rVert}\newcommand{\abs}[1]{\lvert #1\rvert}
\newcommand{\bigo}{\mathcal{O}}\newcommand{\id}{\mathbb{I}}\newcommand{\dg}{^{\dagger}}
\newcommand{\R}{\mathbb{R}}\newcommand{\C}{\mathbb{C}}\newcommand{\N}{\mathbb{N}}
\newcommand{\Z}{\mathbb{Z}}\newcommand{\E}{\mathbb{E}}\newcommand{\Prob}{\mathbb{P}}
\newcommand{\Hilb}{\mathcal{H}}\newcommand{\ii}{\mathrm{i}}\newcommand{\dd}{\mathrm{d}}
\newcommand{\ee}{\mathrm{e}}\newcommand{\QFT}{\mathrm{QFT}}\newcommand{\CNOT}{\mathrm{CNOT}}
\newcommand{\Uf}{U_f}\newcommand{\defeq}{:=}
\newcommand{\Xgate}{\mathsf{X}}\newcommand{\Ygate}{\mathsf{Y}}
\newcommand{\Zgate}{\mathsf{Z}}\newcommand{\Hgate}{\mathsf{H}}$

# Quantum Singular Value Transformation and Matrix Functions

**Learning objectives.**

-   Extend QSP to the singular value transformation of general matrices.

-   State the real-polynomial QSVT theorem.

-   Realize matrix functions (inverse, exponential, filters) via QSVT.

-   Connect QSVT to LCU as two routes to matrix functions.

Qubitization and quantum signal processing (that chapter) taught us to apply a
bounded degree-$d$ polynomial to the *eigenvalues* of a Hermitian block-encoded
matrix using $\bigo(d)$ queries and unit subnormalization. But the input models of
that chapter routinely produce block encodings of *non-Hermitian* matrices---a
sparse coefficient matrix of a linear system, a nonsymmetric generator of a
differential equation, a rectangular data matrix---for which "eigenvalue" is not
even the right notion. The correct object is the *singular value decomposition*,
and the correct generalization of the eigenvalue transformation is the
*quantum singular value transformation* (QSVT), due to Gilyén, Su, Low, and
Wiebe: a single degree-$d$ polynomial applied to the *singular values* of $A$,
again with $\bigo(d)$ queries to the block encoding $U_A$ and its adjoint $U_A\dg$.
Remarkably, the circuit is *identical* to the general-encoding qubitization of
that section---the alternation of $U_A$ and $U_A\dg$ that we were forced into by
non-Hermiticity is exactly what threads a degree through the singular values---so QSVT
requires almost no new machinery beyond a change of viewpoint from eigenvectors to
left/right singular vectors (that section). Once the transformation is set up we
record the working theorem for real target polynomials (that section), and then
harvest the three applications that make QSVT the "grand unification" of quantum
algorithms: matrix inversion $x\mapsto1/(\kappa x)$ (the engine of the quantum linear
systems problem, that chapter), Hamiltonian simulation $x\mapsto\ee^{-\ii xt}$ via
the Jacobi--Anger expansion (that chapter), and spectral filtering by an
approximate sign or rectangle function (that section). We then compare QSVT with
the LCU route of that chapter---coherent versus $\ell^1$-weighted, singular value
versus eigenvalue---to see when each is preferable (that section), and close with
the practical question of computing the phase factors (that section). Throughout,
$A\in\C^{N\times N}$ with $N=2^n$ is accessed by a $(1,m)$-block encoding
$U_A\in\mathrm{BE}_{1,m}(A)$, so that $\norm{A}\le1$ and all singular values lie in
$[0,1]$; we reuse the QSP theorem (that theorem) and its real-polynomial
corollary (that corollary) without reproof.

## From eigenvalue to singular value transformation

##### Singular value decomposition and the three matrix functions.

Let $A\in\C^{N\times N}$ have singular value decomposition
$$A=W\Sigma V\dg=\sum_{i\in[N]}\sigma_i\,\ket{w_i}\!\bra{v_i},
  \qquad 0\le\sigma_i\le1,$$
where $\{\ket{v_i}\}$ (right singular vectors) and $\{\ket{w_i}\}$ (left singular
vectors) are orthonormal bases and $\Sigma=\diag(\sigma_0,\dots,\sigma_{N-1})$.
Equivalently $A\ket{v_i}=\sigma_i\ket{w_i}$ and $A\dg\ket{w_i}=\sigma_i\ket{v_i}$.
Given a scalar function $f\colon\R\to\C$, there are *three* natural matrix
functions built from the equation above, depending on which singular basis is used
on the two sides.

**Definition (Generalized matrix functions).**

With $A$ as in the equation above and $f(\Sigma)\defeq\diag(f(\sigma_0),\dots,
f(\sigma_{N-1}))$, define
$$f^{(\mathrm{SV})}(A)\defeq W f(\Sigma) V\dg,\qquad
  f^{\triangleright}(A)\defeq W f(\Sigma) W\dg,\qquad
  f^{\triangleleft}(A)\defeq V f(\Sigma) V\dg .$$
We call $f^{(\mathrm{SV})}(A)$ the (two-sided) *singular value transformation*,
and $f^{\triangleright},f^{\triangleleft}$ the *left* and *right* generalized
matrix functions; all three are loosely referred to as singular value
transformations.

The three coincide when $A$ is Hermitian and $f$ has definite parity; in general they
differ, and QSVT will produce $f^{(\mathrm{SV})}(A)$ for *odd* $f$ and
$f^{\triangleleft}(A)$ (equivalently $f^{\triangleright}(A)$) for *even* $f$. Two
identities record their basic algebra.

**Proposition (Relations among the generalized functions).**

$f^{(\mathrm{SV})}(A\dg)=\bigl(f^{(\mathrm{SV})}(A)\bigr)\dg$,
$f^{\triangleleft}(A)=f^{\triangleright}(A\dg)$, and, for even $f$,
$$f^{\triangleleft}(A)=f\!\bigl(\sqrt{A\dg A}\bigr),\qquad
  f^{\triangleright}(A)=f\!\bigl(\sqrt{AA\dg}\bigr).$$

*Proof.*

The first two are immediate from the equation above and the SVD of $A\dg=V\Sigma
W\dg$. For the equation above, note $A\dg A=V\Sigma^2 V\dg$, so
$\sqrt{A\dg A}=V\Sigma V\dg$ is Hermitian with eigenvalue decomposition sharing the
right singular vectors and eigenvalues $\sigma_i$; hence
$f(\sqrt{A\dg A})=Vf(\Sigma)V\dg=f^{\triangleleft}(A)$, and symmetrically for
$AA\dg=W\Sigma^2W\dg$.

##### Qubitization of a general matrix, in the singular basis.

The construction of that section carries over verbatim once eigenvectors are
replaced by singular vectors. Applying $U_A\in\mathrm{BE}_{1,m}(A)$ to the flagged
right singular state gives, with $\Pi=\proj{0^m}\otimes\id$,
$$U_A\ket{0^m}\ket{v_i}=\sigma_i\ket{0^m}\ket{w_i}
     +\sqrt{1-\sigma_i^2}\,\ket{\perp_i'},\qquad \Pi\ket{\perp_i'}=0,$$
because $\Pi U_A\Pi=\proj{0^m}\otimes A$ and $A\ket{v_i}=\sigma_i\ket{w_i}$. Since
$U_A\dg\in\mathrm{BE}_{1,m}(A\dg)$ and $A\dg\ket{w_i}=\sigma_i\ket{v_i}$, the same step
run backward defines a companion state $\ket{\perp_i}$ via
$U_A\dg\ket{0^m}\ket{w_i}=\sigma_i\ket{0^m}\ket{v_i}+\sqrt{1-\sigma_i^2}\,\ket{\perp_i}$.
Set
$$\mathcal B_i=\{\ket{0^m}\ket{v_i},\ket{\perp_i}\},\qquad
  \mathcal B_i'=\{\ket{0^m}\ket{w_i},\ket{\perp_i'}\},$$
spanning two-dimensional subspaces $\mathcal H_i,\mathcal H_i'$. Exactly as in
that lemma,
$$^{\mathcal B_i'}_{\mathcal B_i}
  =\begin{pmatrix}\sigma_i&\sqrt{1-\sigma_i^2}\\[2pt]
    \sqrt{1-\sigma_i^2}&-\sigma_i\end{pmatrix}
  =[U_A\dg]^{\mathcal B_i}_{\mathcal B_i'},\qquad
  [Z_\Pi]_{\mathcal B_i}=[Z_\Pi]_{\mathcal B_i'}=\begin{pmatrix}1&0\\0&-1\end{pmatrix},$$
so $U_A$ maps $\mathcal H_i\to\mathcal H_i'$ and $U_A\dg$ maps $\mathcal H_i'\to
\mathcal H_i$, and $Z_\Pi=2\Pi-\id$ fixes the shared flagged vector while flipping the
perpendicular one. The qubitization iterate $\widetilde O=U_A\dg Z_\Pi U_A Z_\Pi$
therefore restricts to $O(\sigma_i)^2$ on $\mathcal H_i$, with $O(\sigma_i)$ the planar
rotation of the equation above. Reading off the $(1,1)$-entries of the powers gives
the singular value analogue of that theorem.

**Theorem (Chebyshev singular value transformation).**

Let $U_A\in\mathrm{BE}_{1,m}(A)$ and $Z_\Pi=2\Pi-\id$ with $\Pi=\proj{0^m}\otimes\id$.
Then for every $k\ge1$,
$$\bigl(U_A\dg Z_\Pi U_A Z_\Pi\bigr)^k
    \in\mathrm{BE}_{1,m}\bigl(T_{2k}^{\triangleleft}(A)\bigr),\qquad
  U_A Z_\Pi\bigl(U_A\dg Z_\Pi U_A Z_\Pi\bigr)^k
    \in\mathrm{BE}_{1,m}\bigl(T_{2k+1}^{(\mathrm{SV})}(A)\bigr),$$
where $T_{2k}^{\triangleleft}(A)=V T_{2k}(\Sigma)V\dg$ and
$T_{2k+1}^{(\mathrm{SV})}(A)=W T_{2k+1}(\Sigma)V\dg$.

*Proof.*

By the equation above the composite $\widetilde O$ acts on $\mathcal H_i$ as
$\bigl([U_A]^{\mathcal B_i'}_{\mathcal B_i}Z\bigr)^2=O(\sigma_i)^2$, whose $k$-th power
$O(\sigma_i)^{2k}$ has $(1,1)$-entry $T_{2k}(\sigma_i)$ (that lemma); since
the $\mathcal H_i$ are orthogonal and $\{\ket{0^m}\ket{v_i}\}$ span the range of $\Pi$,
$\Pi\widetilde O^k\Pi=\proj{0^m}\otimes\sum_i T_{2k}(\sigma_i)\proj{v_i}
=\proj{0^m}\otimes V T_{2k}(\Sigma)V\dg$, the even (left/right) case with base vector
$\ket{v_i}$ mapped to itself. Prepending one $U_A Z_\Pi$ carries
$\mathcal H_i\to\mathcal H_i'$; the surviving flagged block is the $(1,1)$-entry of
$O(\sigma_i)^{2k+1}$, namely $T_{2k+1}(\sigma_i)$, and the base vector is now
$\ket{0^m}\ket{w_i}$, giving $\sum_i T_{2k+1}(\sigma_i)\ket{w_i}\!\bra{v_i}=
W T_{2k+1}(\Sigma)V\dg$.

Two structural facts are worth isolating. First, *parity is forced*: an even
degree returns a one-sided transformation ($V\!\cdots V\dg$), an odd degree returns the
two-sided $W\!\cdots V\dg$. This is not a defect but the honest reflection of the fact
that singular values are nonnegative and a matrix function must respect the two
singular bases. Second, when we only care about $f^{(\mathrm{SV})}(A)$ for a general
$f$, we approximate an *odd* extension of $f$ (odd about the origin) so that the
polynomial has odd parity; for $f^{\triangleleft}(A)$ we approximate an *even*
extension. Because all $\sigma_i\ge0$, evaluating the extended polynomial on $[0,1]$
recovers $f$ on the singular values regardless of the extension's behavior on $[-1,0)$.

##### The Hermitian special case.

When $A$ is Hermitian with eigendecomposition $A=\sum_i\lambda_i\proj{u_i}$, the SVD is
$\sigma_i=\abs{\lambda_i}$, $\ket{v_i}=\ket{u_i}$, $\ket{w_i}=\operatorname{sgn}
(\lambda_i)\ket{u_i}$. For an *even* polynomial $P$, $P(\lambda_i)=P(\sigma_i)$,
so $P^{\triangleleft}(A)=P(A)$; for an *odd* polynomial,
$P(\lambda_i)=\operatorname{sgn}(\lambda_i)P(\sigma_i)$, so
$P^{(\mathrm{SV})}(A)=P(A)$. Thus *for a definite-parity polynomial the QSVT and
the QET of that chapter produce the same operator*: eigenvalue transformation and
singular value transformation are two views of one circuit. The distinction becomes
real only when $A$ is non-Hermitian, or when $P$ has indefinite parity (then the two
objects genuinely differ and cannot be produced by a single definite-parity QSVT
sweep).

## Real-polynomial QSVT

Lifting the scalar QSP identity to a matrix is now automatic: replace the signal
rotation $O(\lambda)$ by the qubitization iterate built from $U_A,U_A\dg$, and each
$Z$-rotation $\ee^{\ii\phi_j Z}$ by the flag rotation $\ee^{\ii\phi_j Z_\Pi}$. Because
every factor preserves the subspaces $\mathcal H_i$ and acts there as the corresponding
$2\times2$ QSP factor at $\lambda=\sigma_i$, the flagged block of the whole product is
the QSP polynomial evaluated singular value by singular value. Concretely, for even
degree $d$,
$$U_\Phi=\ee^{\ii\phi_0 Z_\Pi}\!\!\prod_{j=1}^{d/2}\!\Bigl[(U_A\dg Z_\Pi)\,
     \ee^{\ii\phi_{2j-1}Z_\Pi}(U_A Z_\Pi)\,\ee^{\ii\phi_{2j}Z_\Pi}\Bigr]$$
block-encodes $P^{\triangleleft}(A)$ for an even $P$, and for odd degree $d$ the
analogous product ending in one extra $U_A Z_\Pi$ block-encodes $P^{(\mathrm{SV})}(A)$
for an odd $P$. As in that section, the interleaved reflections $Z_\Pi$ are
absorbed into the flag rotations by a $\pm\pi/2$ shift of the phases (the
"$C$-convention"), so the physical circuit is: $d$ alternating calls to $U_A/U_A\dg$,
$d+1$ controlled flag rotations $\mathrm{CR}(\phi_j)$ (each an
$m$-qubit-controlled single-qubit $Z$-rotation), and one extra ancilla. This is
that theorem transported to the block encoding, and it holds for the full class
of complex definite-parity $P$ of that theorem. In practice we want a
*real* target, extracted by the same $\tfrac12(P+P^*)$ trick as in QSP.

**Theorem (QSVT with real polynomials).**

Let $A\in\C^{N\times N}$ be encoded by $U_A\in\mathrm{BE}_{1,m}(A)$, and let
$P\in\R[x]$ have degree $d$, parity $d\bmod2$, and $\abs{P(x)}\le1$ on $[-1,1]$ (the
conditions of that corollary). Then there is a phase sequence
$\Phi\in\R^{d+1}$ such that the QSVT circuit $U_\Phi$ (that figure) is a
$(1,m+1)$-block encoding of $P^{(\mathrm{SV})}(A)=W P(\Sigma) V\dg$ when $d$ is odd,
and of $P^{\triangleleft}(A)=V P(\Sigma) V\dg$ when $d$ is even. The circuit uses
$U_A,U_A\dg$, an $m$-qubit controlled-[not]{.smallcaps}, and single-qubit rotation gates a
total of $\bigo(d)$ times, with one additional ancilla qubit.

*Proof (Proof sketch).*

By the cosine--sine decomposition of that section, in the singular basis $U_\Phi$
is a direct sum of the $2\times2$ QSP unitaries $U_\Phi(\sigma_i)$ over $i$ (plus inert
$1\times1$ blocks when the flagged and unflagged ranks differ). By
that theorem the $(1,1)$-entry of $U_\Phi(\sigma_i)$ is a complex $P$ of the
stated degree and parity, so the flagged block of $U_\Phi$ is $\sum_i P(\sigma_i)
(\text{base})_i$, i.e. $P^{(\mathrm{SV})}$ (odd $d$) or $P^{\triangleleft}$ (even $d$).
To extract the *real* part $P_{\mathrm{Re}}=\operatorname{Re}P$ without a second
ancilla, sandwich $U_\Phi$ between Hadamards on a control qubit and negate all phases in
the $\ket1$ branch: the $\ket1$ branch block-encodes $(U_\Phi)^*$, whose flagged block
is $P^*(\Sigma)$, and the Hadamard interference leaves $\tfrac12(P+P^*)(\Sigma)=
P_{\mathrm{Re}}(\Sigma)$ in the flagged $\ket0$ outcome. Since running $U_{-\Phi}$ is
the same circuit with the signal qubit initialized in $\ket1$, no new gates are needed;
the phase factors $\Phi$ exist by that corollary. Query count and ancilla count
are read off the circuit.

<figure>

<figcaption>Schematic of the real-polynomial QSVT circuit <span class="math inline"><em>U</em><sub><em>Φ</em></sub></span> of
that theorem. It is the general-encoding qubitization of that section
with interleaved flag rotations <span class="math inline">CR(<em>ϕ</em><sub><em>j</em></sub>) ≡ ⅇ<sup>ⅈ<em>ϕ</em><sub><em>j</em></sub><em>Z</em><sub><em>Π</em></sub></sup></span> (in the
<span class="math inline"><em>C</em></span>-convention) and one extra control ancilla to extract <span class="math inline">Re <em>P</em></span>.</figcaption>
</figure>

##### Matrix dilation: QSVT from QET.

There is an instructive second route that reduces QSVT to the Hermitian QET of
that chapter. Form the Hermitian *dilation*
$$D(A)=\begin{pmatrix}0&A\dg\\ A&0\end{pmatrix},\qquad
  \ket{z_i^{\pm}}=\tfrac1{\sqrt2}\bigl(\ket0\ket{v_i}\pm\ket1\ket{w_i}\bigr),\qquad
  D(A)\ket{z_i^{\pm}}=\pm\sigma_i\ket{z_i^{\pm}},$$
so the eigenvalues of $D(A)$ are $\pm\sigma_i$. A block encoding $U_{D(A)}$ is built
from a controlled $U_A/U_A\dg$ with one extra ancilla. Applying the QET of $D(A)$ with a
polynomial $f$ yields, in the $2\times2$ ancilla block,
$$f\bigl(D(A)\bigr)=\begin{pmatrix}f^{\triangleleft}_{\mathrm{even}}(A)&
     f^{(\mathrm{SV})}_{\mathrm{odd}}(A\dg)\\[2pt]
     f^{(\mathrm{SV})}_{\mathrm{odd}}(A)&f^{\triangleright}_{\mathrm{even}}(A)\end{pmatrix},$$
i.e. the even part of $f$ produces the one-sided functions on the diagonal and the odd
part produces the two-sided $f^{(\mathrm{SV})}$ off-diagonal. In particular, for odd
$f$, $f(D(A))\ket0\ket\psi=\ket1\,f^{(\mathrm{SV})}(A)\ket\psi$: the signal qubit
deterministically flips to $\ket1$ and the system holds $f^{(\mathrm{SV})}(A)\ket\psi$.
The dilation viewpoint makes the parity bookkeeping of the equation above
transparent and is how one often analyzes QLSP (that chapter); the direct circuit of
that theorem is more economical because it never doubles the matrix.

## Applications: inversion, exponential, filtering

Every application below is an instance of one recipe: pick the target $f$, find a
bounded real polynomial $P$ of odd (or even) parity approximating $f$ on the relevant
sub-interval of $[0,1]$, and invoke that theorem. The degree $d$ is the query
cost, so the game is polynomial approximation theory.

### Matrix inversion: $f(x)=1/(\kappa x)$ {#matrix-inversion-fx1kappa-x .unnumbered}

Let $A$ be invertible with condition number $\kappa=\norm{A}\,\norm{A^{-1}}$; after the
normalization $\norm{A}\le1$, the singular values lie in $[1/\kappa,1]$. Then
$A^{-1}=V\Sigma^{-1}W\dg=g^{(\mathrm{SV})}(A\dg)$ with $g(x)=1/x$, an odd function.
Since $1/x$ is unbounded and singular at $0$, we cannot approximate it on all of
$[-1,1]$; we only need it on the spectral support $[1/\kappa,1]$ (and, by oddness, on
$[-1,-1/\kappa]$). Choose an odd polynomial $P$ with
$$\Bigl|P(x)-\frac{1}{\eta\kappa x}\Bigr|\le\epsilon
  \quad\text{for all } x\in[-1,-\tfrac1\kappa]\cup[\tfrac1\kappa,1],
  \qquad \abs{P(x)}\le1 \text{ on } [-1,1],$$
where the subnormalization $\eta\ge1$ (say $\eta=4/3$) merely rescales so that
$\abs{P}\le1$ globally, as that corollary requires. A polynomial of degree
$$d=\bigo\!\Bigl(\kappa\log\tfrac1\epsilon\Bigr)$$
achieves the equation above; the linear scaling in $\kappa$ is essentially
optimal and matches the classical intuition that resolving $1/x$ down to $x=1/\kappa$
requires resolution $\sim\kappa$. that theorem then block-encodes
$P^{(\mathrm{SV})}(A\dg)\approx\tfrac1{\eta\kappa}A^{-1}$ with $\bigo(\kappa\log
(1/\epsilon))$ queries. This is the primitive that solves the quantum linear systems
problem: prepare $\ket b$, apply the block encoding, and amplitude-amplify the flagged
outcome; the full complexity accounting (including the overlap-dependent success
probability and the $\bigo(\kappa)$ rounds of amplitude amplification) is the subject
of that chapter. The worked example at the end of the section constructs the
approximating polynomial explicitly.

### Hamiltonian simulation: $f(x)=\ee^{-\ii x t}$ {#hamiltonian-simulation-fxee-ii-x-t .unnumbered}

For a Hermitian $A=H$ with $\norm H\le1$, time evolution is the eigenvalue
transformation $\ee^{-\ii Ht}=\sum_i\ee^{-\ii\lambda_i t}\proj{u_i}$. Split into real
and imaginary parts $\cos(Ht)$ and $\sin(Ht)$, each of definite parity, and approximate
the scalars $\cos(tx),\sin(tx)$ on $[-1,1]$ by their truncated *Jacobi--Anger*
expansions,
$$\cos(tx)=J_0(t)+2\sum_{k\ge1}(-1)^{k}J_{2k}(t)\,T_{2k}(x),\qquad
  \sin(tx)=2\sum_{k\ge0}(-1)^{k}J_{2k+1}(t)\,T_{2k+1}(x),$$
where $J_\nu$ are Bessel functions of the first kind. Because $\abs{J_\nu(t)}$ decays
super-exponentially once $\nu\gtrsim \ee t/2$, truncating at
$$d=\bigo\!\Bigl(t+\frac{\log(1/\epsilon)}{\log(\ee+\log(1/\epsilon)/t)}\Bigr)
   =\bigo\!\Bigl(t+\log\tfrac1\epsilon\Bigr)$$
gives a uniform $\epsilon$-approximation. Each truncated series is a bounded real
even/odd polynomial (rescaled by $\tfrac12$ so the sum of $\abs{\text{coefficients}}$
does not exceed $1$), so QET realizes $\cos(Ht)$ and $\sin(Ht)$, and one LCU combines
them into $\ee^{-\ii Ht}$. The additive $t$ term is the optimal "no fast-forwarding"
scaling and the additive $\log(1/\epsilon)$ is the optimal precision dependence---QSVT
is thus an optimal Hamiltonian-simulation primitive, developed in full in
that chapter. (For a general non-Hermitian block encoding one can use the same
Jacobi--Anger polynomials on the singular values, or the QETU circuit of the
reading, which applies $F(\cos(H/2))$ directly from the evolution oracle $\ee^{-\ii
H}$.)

**Example (Degree of a Hamiltonian-simulation polynomial, with numbers).**

Fix an evolution time $t=50$ (in units where $\norm H\le1$) and target error
$\epsilon=10^{-6}$. The Jacobi--Anger coefficients $J_\nu(t)$ in
the equation above stay $\bigo(1)$ for $\nu\lesssim t$ and begin a
super-exponential decay once $\nu\gtrsim\tfrac{\ee}{2}t\approx68$; a handful of
further terms buys each additional digit. Truncating the series at
$$d\approx\tfrac{\ee}{2}\,t+\bigo\!\bigl(\log\tfrac1\epsilon\bigr)\approx 68+\text{(a
  few)}\approx 90$$
Chebyshev terms already gives $\norm{\cos(Ht)-P_{\cos}(H)}\le\epsilon$ (and likewise
for $\sin$). So QSVT block-encodes $\ee^{-\ii Ht}$ at $t=50$ to six digits with only
$\sim\!90$ queries to the block encoding of $H$---linear in $t$ and essentially
independent of the dimension $2^n$. For contrast, a first-order product formula
(that chapter) needs $\bigo(t^2/\epsilon)$ steps to the same accuracy,
astronomically more at these parameters; this gap is the practical payoff of the
optimal $\bigo(t+\log(1/\epsilon))$ scaling the equation above.

### Filtering: eigenstate and energy filters {#filtering-eigenstate-and-energy-filters .unnumbered}

Many algorithms need a *spectral filter*: a matrix that is $\approx1$ on part of
the spectrum and $\approx0$ elsewhere, used to project (approximately) onto a target
eigenspace, prepare ground states, or estimate energies. The building block is a
polynomial approximation to the *sign function* $\operatorname{sgn}(x)$, which in
turn gives a shifted step (rectangle). Concretely, an odd polynomial $P_{\mathrm{sgn}}$
with
$$\bigl|P_{\mathrm{sgn}}(x)-\operatorname{sgn}(x)\bigr|\le\epsilon
  \quad\text{for } \abs{x}\in[\Delta,1],\qquad \abs{P_{\mathrm{sgn}}}\le1 \text{ on }
  [-1,1],$$
exists with degree $d=\bigo\!\bigl(\Delta^{-1}\log(1/\epsilon)\bigr)$, where
$2\Delta$ is the width of the "dead zone" around $0$ that we decline to resolve
(equivalently the spectral gap we assume). Such $P_{\mathrm{sgn}}$ is built from a
rescaled error function $\operatorname{erf}(kx)$ with $k\sim\Delta^{-1}
\sqrt{\log(1/\epsilon)}$, itself expanded in Chebyshev polynomials. From the sign we
assemble:

-   an *approximate rectangle* on $[a,b]$ via
    $R(x)=\tfrac12\bigl[P_{\mathrm{sgn}}(x-a)-P_{\mathrm{sgn}}(x-b)\bigr]$ (an even or
    indefinite-parity polynomial, handled by the parity splitting of
    that section);

-   an *eigenstate filter* $R(H)$ that suppresses eigencomponents outside
    $[a,b]$, so that $R(H)\ket\psi$ retains only the target band---the basis of
    ground-state preparation and of the linear-systems eigenvalue filter of Lin--Tong;

-   a *fixed-point amplitude amplification / uniform singular value
    amplification* primitive, obtained by choosing $P_{\mathrm{sgn}}$ to send a small
    known overlap $a\ge\delta$ to $\approx1$.

**Proposition (Fixed-point amplitude amplification via QSVT).**

Let $U_A$ be an $n$-qubit unitary and $\Pi'$ a projector with
$\Pi'U_A\ket{\varphi_0}=a\ket\psi$, $a\ge\delta>0$. Then there is an $(n+1)$-qubit QSVT
circuit $U_\Phi$ with global-phase-invariant distance
$D_p\bigl(\ket0\ket\psi,\,U_\Phi\ket0\ket{\varphi_0}\bigr)\le\epsilon$, using
$U_A,U_A\dg$ and controlled reflections $\bigo\!\bigl(\delta^{-1}\log(1/\epsilon)\bigr)$
times.

*Proof.*

In the bases $\mathcal B=\{\ket{\varphi_0},\dots\}$, $\mathcal B'=\{\ket\psi,\dots\}$,
$U_A$ has the flagged $1\times1$ "singular value" $a=\bra\psi U_A\ket{\varphi_0}$
(cf. the Grover/reflection picture of that chapter). Choose an odd real polynomial
$F$ with $\abs{F(x)-1}\le\epsilon^2/2$ for $x\in[\delta,1]$ and $\abs F\le1$; by
the equation above this is a shifted sign approximation with degree
$\bigo(\delta^{-1}\log(1/\epsilon))$. QSVT then block-encodes
$F(a)\ket\psi\!\bra{\varphi_0}$, and the output overlap with $\ket0\ket\psi$ is
$\abs{F(a)}\ge1-\epsilon^2/2$, whence $D_p=\sqrt{2(1-\abs{F(a)})}\le\epsilon$. Unlike
standard amplitude amplification this never overshoots and needs only the lower bound
$\delta$, not the exact value of $a$.

Choosing $A=R_{\psi_0}$ (the Grover diffusion reflection) and $a=1/\sqrt N$ recovers
unstructured search: $\deg F=\bigo(\sqrt N\log(1/\epsilon))$ finds the marked state to
precision $\epsilon$ with the Grover speedup and no overshoot. The same construction
with $A$ a general block encoding *amplifies its singular values uniformly*
toward $\norm A$ (uniform singular value amplification), a subroutine we reuse when
converting a loosely subnormalized block encoding to a tight one.

**Example (Approximating $1/x$ on $[1/\kappa,1]$).**

We make the equation above concrete. A standard construction starts from the
integral identity, valid for $x\neq0$,
$$\frac1x=\int_0^\infty \ee^{-x^2 t^2/2}\,\sqrt{\tfrac2\pi}\;\operatorname{sgn}(x)\,
     x\,\dd(\text{\dots})$$
but the cleaner route (Childs--Kothari--Somma) is: on $\abs x\ge1/\kappa$ write
$\tfrac1x=\tfrac{1-(1-x^2)^{b}}{x}$ up to error $(1-\kappa^{-2})^b\le\ee^{-b/\kappa^2}$,
choose $b=\bigo(\kappa^2\log(1/\epsilon))$, and note that
$\tfrac{1-(1-x^2)^b}{x}$ is an *odd* polynomial of degree $2b-1$ in $x$ that is
bounded by $\bigo(\kappa)$ on $[-1,1]$. Truncating its Chebyshev expansion and
discarding the negligible high-degree tail reduces the degree to $\bigo(\kappa\log
(1/\epsilon))$ while keeping the error $\epsilon$ on $[1/\kappa,1]$; finally divide by
$\eta\kappa$ (with $\eta=4/3$) so that $\abs{P}\le1$ on all of $[-1,1]$. The resulting
odd $P$ satisfies the equation above, and by that theorem the QSVT
circuit block-encodes $P^{(\mathrm{SV})}(A\dg)=V P(\Sigma)V\dg\approx\tfrac1{\eta\kappa}
A^{-1}$ using $\bigo(\kappa\log(1/\epsilon))$ queries. This is precisely the polynomial
whose phase factors one obtains from a QSP solver (that section); the companion
notebook plots $P$ against $1/(\eta\kappa x)$ and checks the error on $[1/\kappa,1]$.

## QSVT versus LCU

We now have two general-purpose ways to realize a matrix function $f(A)$ from a block
encoding: build it as a *linear combination of unitaries* (that chapter), or
apply *QSVT* (this chapter). They are complementary, and the choice matters in
practice.

##### What each transforms.

LCU forms $\sum_\ell c_\ell U_\ell$; when the $U_\ell$
are qubitized Chebyshev blocks $T_k(A)$ this realizes any polynomial
$\sum_k c_k T_k(A)$---an *eigenvalue* transformation for Hermitian $A$ (and only
awkwardly a singular value transformation, via the dilation
the equation above). QSVT natively realizes the *singular value*
transformation of a general matrix, and coincides with the eigenvalue transformation in
the Hermitian definite-parity case. So for a non-Hermitian $A$ with an intrinsic
$f(\sigma_i)$ target, QSVT is the natural tool.

##### Subnormalization: coherent versus $\ell^1$.

This is the decisive
difference. QSVT produces $P(A)$ at *unit* subnormalization (a $(1,m+1)$-block
encoding) using $d$ queries, independent of the size of the coefficients of $P$. LCU
produces $\sum_\ell c_\ell U_\ell$ at subnormalization $\norm{c}_1=\sum_\ell\abs
{c_\ell}$: the success probability of the flagged outcome scales like
$1/\norm{c}_1^2$, so a target polynomial with a large $\ell^1$ coefficient norm (a
generic degree-$d$ polynomial has $\norm{c}_1$ growing with $d$) costs an
$\bigo(\norm{c}_1)$ overhead in amplitude amplification on top of the query count. QSVT
avoids this overhead entirely because qubitization threads the degree through a single
coherent product rather than summing independent branches. Concretely, to realize
$T_d(A)$: LCU of monomials pays $\norm{c}_1\sim2^{d}$; QSVT (or a single qubitization
sweep) pays $1$.

##### Ancillas, generality, and when LCU wins.

QSVT needs one extra ancilla and a
classically precomputed phase sequence; LCU needs $\lceil\log_2 L\rceil$ ancillas for
$L$ terms and a $\mathrm{prep}/\mathrm{select}$ pair. When the function is already a
*short* LCU with small $\norm{c}_1$---for example $\ee^{-\ii Ht}=\cos(Ht)-\ii
\sin(Ht)$ as a two-term combination of QSVT-built factors, or a sum of a handful of
Hamiltonian-simulation terms (LCHS, that chapter)---LCU is simplest and its
$\ell^1$ cost is negligible. LCU is also the right frame when the terms are genuinely
distinct unitaries (a Pauli/LCU Hamiltonian) rather than powers of one block encoding.
The two techniques compose: the standard pattern is "QSVT to build each definite-parity
piece at unit subnormalization, then a low-weight LCU to add the pieces," which is
exactly how $\ee^{-\ii Ht}$ and indefinite-parity $f(A)$ are assembled
(that section).

::: center
                     **QSVT**                                                      **LCU**
  ------------------ ------------------------------------------------------------- ------------------------------------
  transforms         singular values (eigenvalues if Hermitian, definite parity)   eigenvalues
                                                                                   (Chebyshev/monomial basis)
  subnormalization   $1$ (coherent)                                                $\norm{c}_1$ ($\ell^1$ overhead)
  queries            $\bigo(d)$ to $U_A,U_A\dg$                                    $\bigo(L)$ selects $+$ AA overhead
                                                                                   $\bigo(\norm{c}_1)$
  ancillas           $+1$                                                          $+\lceil\log_2 L\rceil$
  best when          high-degree single-$A$ polynomial, non-Hermitian $A$          short sum, distinct
                                                                                   unitaries, small $\norm{c}_1$
:::

## Phase-factor computation

QSVT is only useful if the phase sequence $\Phi\in\R^{d+1}$ realizing a target real
polynomial $P$ can actually be computed. The problem inherits directly from QSP
(that section)---QSVT adds no new phase-finding difficulty, only the deterministic
$C$-convention $\pm\pi/2$ shift of that section---so we recall the three families
and what is specific to matrix-function targets.

##### Direct (root-finding) methods.

Given the real target $P_{\mathrm{Re}}$, one
first *completes* it: find $Q$ with $\abs P^2+(1-x^2)\abs Q^2=1$ (with
$\operatorname{Re}P=P_{\mathrm{Re}}$) by factoring the degree-$2d$ polynomial
$1-P_{\mathrm{Re}}^2$ into Fejér--Riesz square form over its roots, then strips one
degree per phase from the ratio of leading coefficients (the constructive converse of
that theorem). Exact in principle; the root extraction is ill-conditioned for
large $d$ and needs high-precision arithmetic. Prony-type and capitalization variants
improve stability.

##### Optimization-based methods.

Fixing the parity leaves
$\widetilde d=\lceil(d+1)/2\rceil$ free phases; minimize the least-squares mismatch
$$F(\Phi)=\frac1{\widetilde d}\sum_{k=1}^{\widetilde d}
    \Bigl(\operatorname{Re}\bigl[U_\Phi(x_k)_{11}\bigr]-P_{\mathrm{Re}}(x_k)\Bigr)^2,
    \qquad x_k=\cos\!\Bigl(\tfrac{(2k-1)\pi}{4\widetilde d}\Bigr),$$
over $\Phi$ on the positive Chebyshev nodes. Started from the symmetric ansatz
$\Phi_0=(\tfrac\pi4,0,\dots,0,\tfrac\pi4)$ this nonconvex objective behaves almost
convexly near the solution, so quasi-Newton descent converges in double precision even
for $d$ in the thousands---the workhorse for the inversion and filtering polynomials of
that section.

##### Symmetric QSP and the nonlinear Fourier transform.

Restricting to symmetric
sequences $\phi_j=\phi_{d-j}$ makes $U_\Phi(x)_{11}$ automatically real of the correct
parity, halves the unknowns, and turns completion into a well-conditioned
fixed-point/Newton iteration with no explicit $Q$. A recent and conceptually clean
viewpoint recasts phase-factor finding as an (inverse) *nonlinear Fourier
transform*, for which fast, provably stable algorithms exist. In practice one calls a
mature solver (e.g. `QSPPACK`'s optimization routine or a symmetric-QSP Newton
solver) as a black box: input the Chebyshev coefficients of $P_{\mathrm{Re}}$, receive
$\Phi$; the QSVT circuit of that theorem then block-encodes
$P^{(\mathrm{SV})}(A)$ with $\bigo(d)$ queries. For the rest of Part II we therefore
treat "block-encode a bounded degree-$d$ singular value transformation" as a
primitive costing $\bigo(d)$ queries, one ancilla, and $d$ classically precomputed
rotations.

## Exercises {#exercises .unnumbered}

**Exercise.**

*The three generalized functions.* For $A=W\Sigma V\dg$ prove
that result: $f^{(\mathrm{SV})}(A\dg)=(f^{(\mathrm{SV})}(A))\dg$,
$f^{\triangleleft}(A)=f^{\triangleright}(A\dg)$, and, for even $f$,
$f^{\triangleleft}(A)=f(\sqrt{A\dg A})$, $f^{\triangleright}(A)=f(\sqrt{AA\dg})$.
Give an explicit $2\times2$ non-Hermitian $A$ for which $f^{\triangleleft}(A)\neq
f^{\triangleright}(A)$ when $f(x)=x^2$.

**Exercise.**

*Parity and which function you get.* Using that theorem, explain why an
even-degree QSVT sweep produces the one-sided $P^{\triangleleft}(A)=VP(\Sigma)V\dg$
while an odd-degree sweep produces the two-sided $P^{(\mathrm{SV})}(A)=WP(\Sigma)V\dg$.
For a Hermitian $A=\sum_i\lambda_i\proj{u_i}$ and $P$ odd, show
$P^{(\mathrm{SV})}(A)=P(A)$ (eigenvalue transformation), and for $P$ even show
$P^{\triangleleft}(A)=P(A)$; identify where $\operatorname{sgn}(\lambda_i)$ enters.

**Exercise.**

*The Chebyshev singular value block.* Let $A\in\mathrm{BE}_{1,m}(A)$ be a general
(non-Hermitian) encoding. Write out the operator sequences that block-encode
$T_2^{\triangleleft}(A)$ and $T_3^{(\mathrm{SV})}(A)$, count the queries to $U_A$ and to
$U_A\dg$ in each, and verify the counts equal the Chebyshev degree. Using
$T_2(x)=2x^2-1$, express $T_2^{\triangleleft}(A)$ in terms of $A\dg A$.

**Exercise.**

*Matrix dilation.* Verify the equation above: with $\ket{z_i^\pm}=
\tfrac1{\sqrt2}(\ket0\ket{v_i}\pm\ket1\ket{w_i})$, show $D(A)\ket{z_i^\pm}=
\pm\sigma_i\ket{z_i^\pm}$. Deduce the equation above, and conclude that for an
odd $f$, $f(D(A))\ket0\ket\psi=\ket1\,f^{(\mathrm{SV})}(A)\ket\psi$ so that measuring
the signal qubit yields outcome $1$ with certainty.

**Exercise.**

*Inversion degree.* Following that example, show that
$g_b(x)=\tfrac{1-(1-x^2)^b}{x}$ is an odd polynomial of degree $2b-1$, that
$\abs{g_b(x)-1/x}\le\ee^{-b/\kappa^2}/\abs x$ for $\abs x\ge1/\kappa$, and hence that
$b=\bigo(\kappa^2\log(1/\epsilon))$ suffices for pointwise error $\epsilon$ on
$[1/\kappa,1]$. Explain how truncating the Chebyshev expansion of $g_b$ reduces the
effective degree to $\bigo(\kappa\log(1/\epsilon))$, and why the extra $\eta=4/3$ factor
is needed for that corollary.

**Exercise.**

*Jacobi--Anger truncation.* From the equation above and the bound
$\abs{J_\nu(t)}\le(t/2)^\nu/\nu!$, show that truncating the $\cos(tx)$ series after the
$T_{2K}$ term incurs uniform error $\bigo\bigl((t/2)^{2K}/(2K)!\bigr)$ on $[-1,1]$, and
deduce that $K=\bigo(t+\log(1/\epsilon))$ gives error $\epsilon$. Why do we rescale the
truncated series by $\tfrac12$ before feeding it to that theorem?

**Exercise.**

*Fixed-point amplitude amplification.* Prove that result: given
$\Pi'U_A\ket{\varphi_0}=a\ket\psi$ with $a\ge\delta$, and an odd $F$ with
$\abs{F(x)-1}\le\epsilon^2/2$ on $[\delta,1]$ and $\abs F\le1$, show that the QSVT
circuit outputs a state at global-phase-invariant distance
$\sqrt{2(1-\abs{F(a)})}\le\epsilon$ from $\ket0\ket\psi$. Specialize to Grover search
($a=1/\sqrt N$) and read off the degree $\bigo(\sqrt N\log(1/\epsilon))$; contrast with
standard Grover's overshooting behavior.

**Exercise.**

*QSVT versus LCU for $T_d$.* Compare the two ways to block-encode $T_d(A)$ for
Hermitian $A$: (i) a single qubitization/QSVT sweep, and (ii) an LCU of the monomials
$A^0,\dots,A^d$ with $T_d(x)=\sum_k c_k x^k$. Estimate $\norm c_1$ for $T_d$ (recall the
leading coefficient of $T_d$ is $2^{d-1}$), and hence the amplitude-amplification
overhead of the LCU route. State the subnormalization and query count of each and
explain why QSVT is preferred for large $d$.

## Further reading {#further-reading .unnumbered}

-   Lin Lin, *qasc notes*, Ch. 8 (generalized matrix functions, qubitization
    of general matrices and the Chebyshev singular value transformation, the QSVT
    circuit and real-polynomial QSVT theorem, matrix dilation, application to
    linear systems, and Grover/fixed-point amplitude amplification revisited).

-   Lin Lin, *live notes*, Ch. 13 (derivation of QSVT from the cosine--sine
    decomposition, real-polynomial singular value transformation, QSVT beyond the
    computational basis, fixed-point amplitude amplification and uniform singular
    value amplification, Gibbs-state preparation, eigenvalue transformation with
    Hamiltonian-evolution oracles (QETU), and perturbation theory of singular value
    transformations).

-   Dong An, *Quantum Numerical Linear Algebra* tutorial talk (the
    singular value vs. eigenvalue transformation picture, the main result
    "block-encode $p(A)$ with $\bigo(d)$ queries," and the applications
    $f(x)=1/(\kappa x)$ (linear systems), $f(x)=\ee^{-\ii xt}$ (Hamiltonian
    simulation), filtering, and amplitude amplification $f(x)=qx/2$ with
    $\deg p\sim q\log(1/\epsilon)$).

-   A. Gilyén, Y. Su, G. H. Low, and N. Wiebe, *Quantum singular value
    transformation and beyond: exponential improvements for quantum matrix
    arithmetics*, Proc. STOC 2019, 193--204 (arXiv:1806.01838), for the original
    QSVT theorem and the polynomial-approximation corollaries used here; and
    J. Martyn, Z. Rossi, A. Tan, and I. Chuang, *Grand unification of quantum
    algorithms*, PRX Quantum **2**, 040203 (2021), for the unifying picture.

-   A. M. Childs, R. Kothari, and R. D. Somma, *Quantum algorithm for systems
    of linear equations with exponentially improved dependence on precision*, SIAM
    J. Comput. **46**, 1920--1950 (2017), for the explicit $1/x$ polynomial;
    and L. Lin and Y. Tong, *Optimal quantum eigenstate filtering with
    application to solving quantum linear systems*, Quantum **4**, 361 (2020),
    for eigenstate/energy filtering.